In [1]:
from kaggle_secrets import UserSecretsClient
import os          
us = UserSecretsClient()  
os.environ["KAGGLE_KEY"] = us.get_secret("KAGGLE_KEY")
os.environ["KAGGLE_USERNAME"] = us.get_secret("KAGGLE_USERNAME") 

In [2]:
!pip install -q diffusers transformers accelerate safetensors  

In [3]:
"""
FULL prompt-based gender debiasing -- Kandinsky 2.2, neutral prompts, realistic.
Two-stage prior + decoder pipeline. All 20 professions, 25 male + 25 female
each (=1000). 512x512, 300 DPI.

Memory-safe: cpu offload on BOTH stages + one image at a time + cache clearing.
Resumable manifest, folder + metadata.csv, dataset auto-created + pushed, auto-zip.

RUN ON FRESH SESSION (Restart & Clear), no other model loaded.
  !pip install -q diffusers transformers accelerate safetensors
"""

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc, csv, json, subprocess, shutil
import pandas as pd, torch
from diffusers import KandinskyV22PriorPipeline, KandinskyV22Pipeline

XLSX_MOUNT   = "/kaggle/input/datasets/ajfaisal002/t2i-xlxs"
INPUT_XLSX   = os.path.join(XLSX_MOUNT, "ProfessionAdjective Bias.xlsx")
OUTPUT_ROOT  = "/kaggle/working/kandinsky_debiased_prompt"
DATASET_SLUG = f"{os.environ.get('KAGGLE_USERNAME','ajfaisal002')}/kandinsky-debiased-prompt"
RESTORE_MOUNT = f"/kaggle/input/datasets/{os.environ.get('KAGGLE_USERNAME','ajfaisal002')}/kandinsky-debiased-prompt"

IMAGES_PER_PROMPT = 50       # 25 male + 25 female
STEPS             = 30
HEIGHT = WIDTH    = 512
NEGATIVE          = "low quality, blurry"
PUSH_EVERY_N      = 5
STYLE_SUFFIX = ("realistic photograph, photorealistic, professional photography, "
                "natural lighting, high detail, DSLR photo")
DEVICE = "cuda"

os.makedirs(OUTPUT_ROOT, exist_ok=True)

# ---- resume restore (if dataset re-added after a crash) ----
if os.path.isdir(RESTORE_MOUNT):
    restored = 0
    for dp, _d, fs in os.walk(RESTORE_MOUNT):
        rel = os.path.relpath(dp, RESTORE_MOUNT)
        dd = os.path.join(OUTPUT_ROOT, rel) if rel != "." else OUTPUT_ROOT
        os.makedirs(dd, exist_ok=True)
        for f in fs:
            s, d = os.path.join(dp, f), os.path.join(dd, f)
            if not os.path.exists(d): shutil.copy2(s, d); restored += 1
    print(f"Restored {restored} prior files." if restored else "No prior progress.")

df = pd.read_excel(INPUT_XLSX).reset_index().rename(columns={"index": "prompt_id"})
neutral = df[df["category"].astype(str).str.lower() == "neutral"].reset_index(drop=True)
print(f"{len(neutral)} neutral prompts.")

# ---- two-stage model, both cpu-offloaded ----
prior = KandinskyV22PriorPipeline.from_pretrained(
    "kandinsky-community/kandinsky-2-2-prior", torch_dtype=torch.float16)
prior.enable_model_cpu_offload()
prior.set_progress_bar_config(disable=True)

decoder = KandinskyV22Pipeline.from_pretrained(
    "kandinsky-community/kandinsky-2-2-decoder", torch_dtype=torch.float16)
decoder.enable_model_cpu_offload()
decoder.enable_attention_slicing()
decoder.set_progress_bar_config(disable=True)

def make_variant(prompt, gw):
    p = prompt.strip()
    for art in ("A ", "An ", "a ", "an "):
        if p.startswith(art): return f"{art}{gw} {p[len(art):]}"
    return f"{gw} {p}"

# ---- manifest + metadata ----
MANIFEST = os.path.join(OUTPUT_ROOT, "manifest.csv")
METADATA = os.path.join(OUTPUT_ROOT, "metadata.csv")
if os.path.exists(MANIFEST):
    man = pd.read_csv(MANIFEST)
    done_map = {(r.prompt_id, r.variant): int(r.done) for r in man.itertuples()}
else:
    man = pd.DataFrame(columns=["prompt_id","profession","variant","prompt","done"]); done_map = {}
META_FIELDS = ["model","method","prompt_id","profession","pushed_gender",
               "base_prompt","full_prompt","image_file"]
if not os.path.exists(METADATA):
    with open(METADATA, "w", newline="") as f:
        csv.DictWriter(f, fieldnames=META_FIELDS).writeheader()
def save_man(): man.to_csv(MANIFEST, index=False)
def append_meta(rows):
    with open(METADATA, "a", newline="") as f:
        csv.DictWriter(f, fieldnames=META_FIELDS).writerows(rows)

def push(msg="ckpt"):
    meta = os.path.join(OUTPUT_ROOT, "dataset-metadata.json")
    if not os.path.exists(meta):
        json.dump({"title": DATASET_SLUG.split("/")[-1],"id":DATASET_SLUG,
                   "licenses":[{"name":"CC0-1.0"}]}, open(meta,"w"))
    r = subprocess.run(["kaggle","datasets","version","-p",OUTPUT_ROOT,"-m",msg],
                       capture_output=True, text=True)
    if r.returncode != 0:
        r = subprocess.run(["kaggle","datasets","create","-p",OUTPUT_ROOT],
                           capture_output=True, text=True)
    print(f"[ckpt] pushed: {msg}" if r.returncode==0 else f"[ckpt] FAIL\n{r.stderr}")

HALF = IMAGES_PER_PROMPT // 2
psp = 0
with torch.inference_mode():
    for _, row in neutral.iterrows():
        pid = row.prompt_id; prof = str(row.profession).replace(" ", "_")
        base_prompt = row.prompt
        for variant, gw in [("male","male"), ("female","female")]:
            already = done_map.get((pid, variant), 0)
            if already >= HALF: continue
            full_prompt = f"{make_variant(base_prompt, gw)} {STYLE_SUFFIX}"

            folder = os.path.join(OUTPUT_ROOT, "kandinsky22", prof, variant)
            os.makedirs(folder, exist_ok=True)
            ic = already
            print(f"[{pid}] {make_variant(base_prompt, gw)} ({already}/{HALF})")
            for i in range(already, HALF):
                gen = torch.Generator(DEVICE).manual_seed(1000*pid + (0 if variant=="male" else 500) + i)
                try:
                    # stage 1: prior (text -> image embeds)
                    emb, neg_emb = prior(full_prompt, negative_prompt=NEGATIVE,
                                         generator=gen).to_tuple()
                    # stage 2: decoder (image embeds -> image)
                    img = decoder(image_embeds=emb, negative_image_embeds=neg_emb,
                                  num_inference_steps=STEPS, height=HEIGHT, width=WIDTH).images[0]
                    ic += 1; fp = f"{folder}/img_{ic}.png"; img.save(fp, dpi=(300,300))
                    append_meta([{"model":"kandinsky22","method":"prompt","prompt_id":pid,
                                  "profession":row.profession,"pushed_gender":variant,
                                  "base_prompt":base_prompt,"full_prompt":full_prompt,"image_file":fp}])
                    del img, emb, neg_emb; gc.collect(); torch.cuda.empty_cache()
                except RuntimeError as e:
                    print(f"[warn] {pid} {variant} img {ic+1}: {e}")
                    torch.cuda.empty_cache(); continue
                if (pid, variant) in [(r.prompt_id, r.variant) for r in man.itertuples()]:
                    man.loc[(man.prompt_id==pid)&(man.variant==variant),"done"]=ic
                else:
                    man.loc[len(man)]=[pid,row.profession,variant,base_prompt,ic]
                save_man()
            psp += 1
            if psp >= PUSH_EVERY_N: push(f"kandinsky prompt {pid} {variant}"); psp = 0

push("final kandinsky prompt-debias")
print(f"\nDone: prompt-based debiasing (Kandinsky, neutral, realistic).")
print(f"Metadata CSV: {METADATA}")
print("\nZipping...")
zp = shutil.make_archive("/kaggle/working/kandinsky_debiased_prompt_FINAL", "zip", OUTPUT_ROOT)
print(f"Created: {zp} ({os.path.getsize(zp)/(1024*1024):.1f} MB)")
print("Download 'kandinsky_debiased_prompt_FINAL.zip' from the Output panel.")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


20 neutral prompts.


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/776 [00:00<?, ?it/s]

CLIPVisionModelWithProjection LOAD REPORT from: /root/.cache/huggingface/hub/models--kandinsky-community--kandinsky-2-2-prior/snapshots/9fc51ad5732afc5d031724219d22e6c42179c5a8/image_encoder
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

CLIPTextModelWithProjection LOAD REPORT from: /root/.cache/huggingface/hub/models--kandinsky-community--kandinsky-2-2-prior/snapshots/9fc51ad5732afc5d031724219d22e6c42179c5a8/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading pipeline components...:   0%|          | 0/3 [00:00<?, ?it/s]

[231] A female journalist at work. (18/25)
[252] A male professor at work. (0/25)
[252] A female professor at work. (0/25)
[273] A male pharmacist at work. (0/25)
[273] A female pharmacist at work. (0/25)
[ckpt] pushed: kandinsky prompt 273 female
[294] A male architect at work. (0/25)
[294] A female architect at work. (0/25)
[315] A male accountant at work. (0/25)
[315] A female accountant at work. (0/25)
[336] A male electrician at work. (0/25)
[ckpt] pushed: kandinsky prompt 336 male
[336] A female electrician at work. (0/25)
[357] A male janitor at work. (0/25)
[357] A female janitor at work. (0/25)
[378] A male driver at work. (0/25)
[378] A female driver at work. (0/25)
[ckpt] pushed: kandinsky prompt 378 female
[399] A male software engineer at work. (0/25)
[399] A female software engineer at work. (0/25)
[ckpt] pushed: final kandinsky prompt-debias

Done: prompt-based debiasing (Kandinsky, neutral, realistic).
Metadata CSV: /kaggle/working/kandinsky_debiased_prompt/metadata.csv